In [5]:
import os
from dotenv import load_dotenv
from openai import OpenAI

import gradio as gr

In [6]:





load_dotenv(override=True)
openrouter_api = os.getenv("OPENROUTER_API_KEY")
groq_api = os.getenv("GROQ_API_KEY")

if openrouter_api:
    display("Openrouter api found")
else:
    display("OPENROUTER NOT FOUND")
if groq_api:
    display("groq api found")
else:
    display("Groq api not found")

'Openrouter api found'

'groq api found'

In [7]:
groq = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api)
openrouter = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api)

In [8]:
system_message = "You are a helpful assistant"

def message_groq(prompt):
    messages = [{"role":"system","content": system_message},
    {"role":"user", "content": prompt}]

    response = groq.chat.completions.create(
        model = "llama-3.3-70b-versatile",
        messages= messages
    )
    return response.choices[0].message.content

In [10]:
message_groq("What is your model training cutoff?")

'My knowledge cutoff is currently December 2023, but I have access to more recent information via internet search.'

In [11]:
def shout(text):
    print(f"Shout input: {text}")
    return text.upper()

In [12]:
shout("hello")

Shout input: hello


'HELLO'

In [14]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode = "never").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Shout input: hello
Shout input: nisum
Shout input: nisum hell0


In [15]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://bc1039fcda4fd5a055.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Shout input: hello


In [16]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [20]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(share=True,auth=("nisum","123"))

* Running on local URL:  http://127.0.0.1:7864
* Running on public URL: https://6dc985bfa8cb26daad.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Shout input: hello


In [22]:
message_input = gr.Textbox(label="Your message: ", info = "Enter a message to be shouted", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn = shout,
    title= "SHOUT",
    inputs=[message_input],
    outputs=[message_output],
    examples = ["hello",'kite'],
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Shout input: yoooooo


## messaging with ai

In [23]:
message_input = gr.Textbox(label = "Your message:", info="Enter a message for AI MODEL")
message_output = gr.Textbox(label="Response:")

view = gr.Interface(
    fn = message_groq,
    inputs=[message_input],
    outputs=[message_output],
    examples=["What is a Blackhole?", "How many bananas in a dozan ?"],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


## to keep asking followup question without memory

In [24]:
system_message = "You are a helpful assistant that responds in markdown without code blocks.you have a snarky tone"
message_input = gr.Textbox(label = "Your message:", info="Enter a message for AI MODEL")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn = message_groq,
    title="MESSAGE WITH GROQ",
    inputs=[message_input],
    outputs=[message_output],
    examples=["What is a Blackhole?", "Explain blue color to a blind person"],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


## stream back results

In [9]:
def stream_groq(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = groq.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT-4.1-mini", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_groq,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the taste of fish to a person who have never tasted fish"], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


In [10]:
def stream_openrouter(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = groq.chat.completions.create(
        model='openai/gpt-oss-120b:free',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [11]:
def stream_model(prompt, model):
    if model=="GROQ":
        result = stream_groq(prompt)
    elif model=="OPENROUTER":
        result = stream_openrouter(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [35]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["GROQ", "OPENROUTER"], label="Select model", value="GROQ")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs", 
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["How tall is Mt.Everest?","GROQ"],
            ["Where is Mt.Everest Located?","OPENROUTER"]
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


In [12]:
def stream_ollama(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = groq.chat.completions.create(
        model='llama3.1:8b',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [17]:
from website_scraper import scrape_website


In [18]:
link_system_prompt = """
You MUST return ONLY valid JSON.

Format EXACTLY like this:
{
    "links": [
        {"type": "about page", "url": "https://example.com/about"}
    ]
}

Rules:
- No explanation
- No extra text
- No markdown
- Only JSON
"""

In [19]:
def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page: \n"
    prompt += scrape_website(url).text

    if model == "GROQ":
        result = stream_groq(prompt)
    elif model == "OPENROUTER":
        result = stream_openrouter(prompt)
    elif model == "OLLAMA":
        result = stream_ollama(prompt) 
    else:
        raise ValueError("Unknown model")
    yield from result

In [20]:
name_input = gr.Textbox(label = "Company name: ")
url_input = gr.Textbox(label="Landing page url: ")
model_selector = gr.Dropdown(["GROQ","OPENROUTER","OLLAMA"], label = "Select Model", value = "GROQ")
message_output = gr.Markdown(label="Response: ")

view = gr.Interface(
    fn = stream_brochure,
    title = "Brochure_Generator",
    inputs = [name_input, url_input, model_selector],
    outputs = [message_output],
    examples= [
        ['NBA','https://www.nba.com', 'GROQ'],
        ['BMW','https://www.bmwusa.com/all-bmws.html', 'OPENROUTER'],
        ['KTM', 'https://www.ktm.com/en-np/models/naked-bike.html', 'OPENROUTER']
    ],
    flagging_mode="never"
    )
view.launch(
    
)



* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Complete verification in browser, then press Enter...